# Farthest-Point Sampling

**Farthest-Point Sampling (FPS)** is a greedy algorithm for placing $K$ points in a domain so that no region is left unsampled. Starting from an initial seed, each new point is chosen as the one farthest from all already-chosen points:
$$
x_{k+1} = \operatorname{argmax}_{x} \min_{i \leq k} \|x - x_i\|.
$$
The **coverage error** (fill distance) after $K$ samples is
$$
e_K = \max_{x} \min_{i \leq K} \|x - x_i\|.
$$
For uniform random sampling this decays as $O(1/\sqrt{K})$; FPS achieves $O(1/K)$ in 1D, providing a deterministic near-optimal covering.

A **weighted** variant biases sampling toward high-priority regions by multiplying the distance by a spatially varying weight $w(x)$:
$$
x_{k+1} = \operatorname{argmax}_{x}\; w(x) \cdot \min_{i \leq k} \|x - x_i\|.
$$
This notebook demonstrates: (1) the 1D error-rate comparison, (2) 2D unweighted FPS progression, (3) 2D weighted FPS, and (4) an interactive progression viewer.

## Setup

Standard scientific-Python imports. We set a fixed random seed for reproducibility and define the output directory for the snippet image.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from ipywidgets import interact, IntSlider

plt.rcParams["figure.dpi"] = 120
rng = np.random.default_rng(42)

OUT = Path("python/farthest-point")
OUT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------
# Core FPS routine (works for 1D or 2D point arrays)
# ---------------------------------------------------------------
def fps(pts, K, weights=None, seed=0):
    """Run Farthest-Point Sampling on pts (N x d array).
    Returns list of K chosen indices."""
    N = len(pts)
    if weights is None:
        weights = np.ones(N)
    chosen = [seed]
    d = np.full(N, np.inf)
    for _ in range(K - 1):
        p = pts[chosen[-1]]
        d = np.minimum(d, np.sum((pts - p) ** 2, axis=1))
        score = weights * d
        chosen.append(int(np.argmax(score)))
    return chosen

## 1D Coverage Error: Uniform Sampling vs FPS

On the unit interval $[0,1]$ discretized to $N=2000$ points we compare:

- **Uniform random sampling**: $K$ points chosen uniformly at random.  The expected coverage error is $O(1/K)$ for the gaps but has high variance; averaging over 50 trials shows the mean decays as $\approx 1/\sqrt{K}$ in the worst case.
- **FPS**: The $K$ FPS points on a 1D grid optimally space themselves, giving coverage error $\approx 1/(2K)$ — an $O(1/K)$ rate.

We plot both curves on a log–log scale to expose the rate difference clearly.

In [ ]:
N1 = 2000
pts1 = np.linspace(0, 1, N1).reshape(-1, 1)
Ks = np.unique(np.round(np.logspace(0.8, 2.5, 30)).astype(int))

# FPS coverage error
fps_err = []
for K in Ks:
    idx = fps(pts1, K)
    sel = pts1[idx]
    d2 = np.min(np.abs(pts1 - sel.T), axis=1)  # dist to nearest sample
    fps_err.append(d2.max())

# Uniform random sampling coverage error (averaged over trials)
n_trials = 80
uni_err_mean = []
uni_err_std = []
rng2 = np.random.default_rng(7)
for K in Ks:
    errs = []
    for _ in range(n_trials):
        idx = rng2.choice(N1, size=K, replace=False)
        sel = pts1[idx]
        d2 = np.min(np.abs(pts1 - sel.T), axis=1)
        errs.append(d2.max())
    uni_err_mean.append(np.mean(errs))
    uni_err_std.append(np.std(errs))

fps_err = np.array(fps_err)
uni_err_mean = np.array(uni_err_mean)
uni_err_std = np.array(uni_err_std)

# Reference slopes
Kf = Ks.astype(float)
ref1K = 0.6 / Kf          # O(1/K)
ref1sqrtK = 0.5 / np.sqrt(Kf)   # O(1/sqrt(K))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(Ks, fps_err, "o-", color="tab:blue", lw=2, ms=4, label="FPS coverage error")
ax.loglog(Ks, uni_err_mean, "s-", color="tab:orange", lw=2, ms=4, label="Uniform random (mean)")
ax.fill_between(Ks,
                np.clip(uni_err_mean - uni_err_std, 1e-5, None),
                uni_err_mean + uni_err_std,
                color="tab:orange", alpha=0.2, label="Uniform \u00b11 std")
ax.loglog(Ks, ref1K, "--", color="tab:blue", lw=1.2, label=r"$O(1/K)$")
ax.loglog(Ks, ref1sqrtK, "--", color="tab:orange", lw=1.2, label=r"$O(1/\sqrt{K})$")
ax.set_xlabel("Number of samples $K$")
ax.set_ylabel("Coverage error (max gap)")
ax.set_title("1D Coverage Error: Uniform vs FPS")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "error_comparison.png", bbox_inches="tight")
plt.close(fig)

## 2D FPS — Progression Without Weighting

In two dimensions, FPS distributes points across $[-1,1]^2$ so that the Voronoi cells are as equal-area as possible.  We visualize the growing point cloud at $K = 10, 20, 50, 100, 200$ to see how coverage tightens progressively.

In [ ]:
n2 = 200   # grid side
x2 = np.linspace(-1, 1, n2)
X2, Y2 = np.meshgrid(x2, x2)
pts2 = np.c_[X2.ravel(), Y2.ravel()]   # (n2^2, 2)

K_max = 200
all_idx = fps(pts2, K_max)
Kvals = [10, 20, 50, 100, 200]

fig, axes = plt.subplots(1, len(Kvals), figsize=(14, 3.0), constrained_layout=True)
for ax, K in zip(axes, Kvals):
    sel = pts2[all_idx[:K]]
    ax.scatter(sel[:, 0], sel[:, 1], s=12, c="tab:blue", linewidths=0)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")
    ax.set_title(f"K = {K}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("2D FPS progression (uniform)", y=1.02)
fig.savefig(OUT / "fps_2d_progression.png", bbox_inches="tight")
plt.close(fig)

## 2D Weighted FPS

We define a smooth weight image $w : [-1,1]^2 \to [0,1]$ with two bright Gaussian blobs.  The weighting is made strong by taking
$$
w_{\text{eff}}(x) = \bigl(w(x) + 0.01\bigr)^3,
$$
which strongly concentrates sampling in the bright (white) regions while still placing a few points in the dark background.  The weighted FPS criterion becomes
$$
x_{k+1} = \operatorname{argmax}_{x}\; w_{\text{eff}}(x) \cdot \min_{i \leq k} \|x - x_i\|^2.
$$

In [ ]:
# Build weight image — two Gaussian blobs on dark background
img = (0.9 * np.exp(-((X2 - 0.4) ** 2 + (Y2 + 0.15) ** 2) / 0.06)
       + 0.7 * np.exp(-((X2 + 0.3) ** 2 + (Y2 - 0.35) ** 2) / 0.04)
       + 0.05)
img = np.clip(img / img.max(), 0, 1)

# Strong weighting: cube to really push points into bright regions
w_eff = (img.ravel() + 0.01) ** 3

K_w = 200
all_idx_w = fps(pts2, K_w, weights=w_eff)
Kvals_w = [10, 20, 50, 100, 200]

fig, axes = plt.subplots(1, len(Kvals_w) + 1, figsize=(16, 3.2), constrained_layout=True)
# First panel: weight image alone
axes[0].imshow(img, origin="lower", extent=[-1, 1, -1, 1], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Weight image")
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_aspect("equal")
# Progression panels
for ax, K in zip(axes[1:], Kvals_w):
    sel = pts2[all_idx_w[:K]]
    ax.imshow(img, origin="lower", extent=[-1, 1, -1, 1], cmap="gray", vmin=0, vmax=1, alpha=0.55)
    ax.scatter(sel[:, 0], sel[:, 1], s=14, c="tab:red", linewidths=0)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")
    ax.set_title(f"K = {K}")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("2D Weighted FPS progression (white = high priority)", y=1.02)
fig.savefig(OUT / "fps_2d_weighted.png", bbox_inches="tight")
plt.close(fig)

## Interactive Progression Viewer

Use the slider below to explore how the FPS point cloud grows from $K=1$ to $K=200$.  The left panel shows uniform FPS; the right shows the weighted variant on the same weight image.

In [ ]:
def show_fps(K=50):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
    # Uniform FPS
    sel_u = pts2[all_idx[:K]]
    axes[0].scatter(sel_u[:, 0], sel_u[:, 1], s=14, c="tab:blue", linewidths=0)
    axes[0].set_xlim(-1.05, 1.05); axes[0].set_ylim(-1.05, 1.05)
    axes[0].set_aspect("equal")
    axes[0].set_title(f"Uniform FPS  K={K}")
    axes[0].set_xticks([]); axes[0].set_yticks([])
    # Weighted FPS
    sel_w = pts2[all_idx_w[:K]]
    axes[1].imshow(img, origin="lower", extent=[-1, 1, -1, 1], cmap="gray", vmin=0, vmax=1, alpha=0.5)
    axes[1].scatter(sel_w[:, 0], sel_w[:, 1], s=14, c="tab:red", linewidths=0)
    axes[1].set_xlim(-1.05, 1.05); axes[1].set_ylim(-1.05, 1.05)
    axes[1].set_aspect("equal")
    axes[1].set_title(f"Weighted FPS  K={K}")
    axes[1].set_xticks([]); axes[1].set_yticks([])
    plt.show()

interact(show_fps, K=IntSlider(min=1, max=200, step=1, value=50,
                                description="K", continuous_update=False));

## Static Snapshot

This cell saves the representative snippet image used in the repository gallery.  It is guarded by `STATIC_SNAPSHOT` so it is skipped during interactive widget sessions.

In [ ]:
STATIC_SNAPSHOT = True

if STATIC_SNAPSHOT:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)

    # Panel 1: 1D error comparison
    ax = axes[0]
    ax.loglog(Ks, fps_err, "o-", color="tab:blue", lw=2, ms=4, label="FPS")
    ax.loglog(Ks, uni_err_mean, "s-", color="tab:orange", lw=2, ms=4, label="Uniform")
    ax.loglog(Ks, ref1K, "--", color="tab:blue", lw=1.2, label=r"$O(1/K)$")
    ax.loglog(Ks, ref1sqrtK, "--", color="tab:orange", lw=1.2, label=r"$O(1/\sqrt{K})$")
    ax.set_xlabel("K")
    ax.set_ylabel("Coverage error")
    ax.set_title("1D Error rates")
    ax.legend(fontsize=7)
    ax.grid(True, which="both", alpha=0.3)

    # Panel 2: 2D uniform FPS at K=100
    ax = axes[1]
    sel_u = pts2[all_idx[:100]]
    ax.scatter(sel_u[:, 0], sel_u[:, 1], s=14, c="tab:blue", linewidths=0)
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")
    ax.set_title("2D FPS (K=100)")
    ax.set_xticks([]); ax.set_yticks([])

    # Panel 3: 2D weighted FPS at K=100
    ax = axes[2]
    sel_w = pts2[all_idx_w[:100]]
    ax.imshow(img, origin="lower", extent=[-1, 1, -1, 1], cmap="gray", vmin=0, vmax=1, alpha=0.55)
    ax.scatter(sel_w[:, 0], sel_w[:, 1], s=14, c="tab:red", linewidths=0)
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")
    ax.set_title("Weighted FPS (K=100)")
    ax.set_xticks([]); ax.set_yticks([])

    fig.savefig(OUT / "snippet.png", bbox_inches="tight")
    plt.close(fig)
    print("Snippet saved.")

## Takeaways

- **FPS achieves $O(1/K)$ coverage error in 1D**, versus $O(1/\sqrt{K})$ for random sampling — a qualitative improvement in spatial regularity.
- In 2D, FPS produces a **well-spread, low-discrepancy point cloud** reminiscent of a Poisson-disk distribution.
- **Weighted FPS** with a strong nonlinear weight ($w^3$) concentrates samples in high-priority regions while maintaining coverage everywhere.
- The greedy algorithm is simple to implement and runs in $O(KN)$ operations.

## Bibliography

- Eldar, Y., Lindenbaum, M., Porat, M., Zeevi, Y. Y. (1997). *The farthest point strategy for progressive image sampling*. IEEE Trans. Image Process.
- Moenning, C., Dodgson, N. A. (2003). *Fast marching farthest point sampling*. University of Cambridge Technical Report.
- Qi, C. R., Yi, L., Su, H., Guibas, L. J. (2017). *PointNet++*. NeurIPS. (uses FPS as a key sub-sampling step.)
- de Berg, M., Cheong, O., van Kreveld, M., Overmars, M. (2008). *Computational Geometry: Algorithms and Applications*, 3rd ed. Springer.